# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 06

### DISEÑO Y ANÁLISIS DE UN EXPERIMENTO A/B

**Semana 6 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía  
**Carrera:** Big Data y Ciencia de Datos  
**Plataforma de trabajo:** Google Colab

### Pregunta orientadora

**¿Qué evidencia estadística y práctica necesitamos antes de implementar una variante B?**

> Este notebook acompaña el trabajo desarrollado durante la clase. No es un informe ni un proyecto para exponer. Las celdas alternan explicación breve, ejecución guiada, interpretación y retos de cierre.


# I. Elemento de la capacidad terminal de la semana

Diseña y analiza un experimento A/B identificando el factor, sus niveles, el control, el tratamiento, la unidad experimental y la variable respuesta. Formula las hipótesis estadísticas, establece la métrica primaria, los guardrails, el nivel de significación, la potencia y el efecto mínimo detectable; finalmente, integra la evidencia estadística con la relevancia práctica para sustentar una decisión.


# II. Seguridad y condiciones de trabajo

- Utilice únicamente Google Colab y las fuentes indicadas por la docente.
- No modifique ni fabrique resultados para forzar una conclusión.
- Ejecute las celdas en el orden propuesto y documente cualquier error o dato ausente.
- No comparta claves, credenciales ni información personal en el notebook.
- Mantenga la estación de trabajo ordenada y no instale software no autorizado.


# III. Fundamento teórico
Un A/B test compara un control A con una variante B mediante asignación aleatoria. La unidad experimental de este caso es el usuario, que conserva su variante durante toda la observación. La aleatorización favorece la comparabilidad en expectativa; no garantiza grupos idénticos.

La respuesta es binaria: compra dentro de los siete días posteriores a la asignación. Todos los usuarios completan esa ventana antes del análisis. La métrica primaria es la proporción de usuarios que compran. Se estudia la diferencia de dos proporciones, no ANOVA.

El MDE es el efecto utilizado para planificar la potencia. El umbral empresarial es la mejora que justificaría interés práctico. Aquí son distintos: MDE de 1,5 pp y umbral empresarial de 1,0 pp. La muestra se dimensiona para detectar el MDE frente a cero, no para garantizar evidencia de superar el umbral empresarial.

Se usa una aproximación normal de Wald para la diferencia. El contraste unilateral al 5 % y el límite inferior unilateral del 95 % utilizan el mismo error estándar. También se muestra un IC bilateral del 95 % como descripción de incertidumbre; no se exige que este último excluya cero para reproducir el contraste unilateral.

La recomendación académica requiere evidencia de superar el umbral empresarial y de mantener el guardrail dentro del margen predefinido. Son criterios más exigentes que obtener p < 0,05 frente a cero. Un resultado no concluyente no demuestra igualdad.


# IV. Normas y buenas prácticas empleadas

- Prerregistro lógico: definir hipótesis, métrica, MDE y regla de decisión antes de observar los resultados.
- Aleatorización: evitar que la asignación dependa de características de las unidades.
- Trazabilidad: conservar la fuente, las transformaciones y los parámetros utilizados.
- Reproducibilidad: ejecutar el análisis mediante código verificable.
- No realizar *peeking*: no detener o prolongar el experimento solo porque el resultado momentáneo conviene.
- No contaminar grupos: una unidad debe recibir únicamente la variante asignada.
- Distinguir asociación, evidencia experimental y decisión práctica.


# V. Recursos
- Computadora, Google Colab y notebook LAB-D6-AEI-PSAYAN-2026-02.ipynb.
- Librerías numpy, pandas, scipy, statsmodels y plotly.
- Caso ficticio NexaCommerce con datos simulados generados en el propio notebook. No requiere descargar un dataset externo.

## Naturaleza de los datos utilizados
Los registros son sintéticos y reproducibles mediante una semilla fija. Representan usuarios de un checkout; no proceden de una empresa real. Las probabilidades del simulador son supuestos académicos, no evidencia observada. No cambie la semilla para buscar un resultado favorable. En un estudio real, los datos procederían de registros del experimento y el plan se documentaría antes de recogerlos.


# VI. Metodología para el desarrollo de la sesión

El laboratorio se desarrolla mediante una secuencia guiada de cinco momentos:

1. **Comprender:** recuperar el vocabulario esencial.
2. **Representar:** traducir el caso a la estructura de un diseño experimental.
3. **Formular:** establecer las hipótesis y la regla de decisión antes de observar resultados.
4. **Ejecutar e interpretar:** correr el código por etapas y explicar cada salida.
5. **Decidir y transferir:** integrar evidencia estadística, umbral empresarial y guardrails; luego resolver retos breves.

No se solicita exposición, informe ni trabajo adicional. La evidencia de aprendizaje se construye durante la clase mediante respuestas breves en las pausas de análisis.


# VII. Procedimiento

## Actividad 1 — Activación conceptual

**Propósito:** reconocer los componentes que deben definirse antes de ejecutar un experimento.

Complete la tabla con sus propias palabras. La pregunta de apoyo permite verificar si la definición es suficiente.

| Concepto | Definición con sus propias palabras | Pregunta de apoyo |
|---|---|---|
| Experimento A/B | Comparación de dos versiones (A y B) en la que las unidades se asignan **al azar** a cada versión y el investigador controla qué recibe cada grupo, para medir el efecto causal de un cambio. | ¿Qué diferencia existe entre comparar dos grupos y experimentar?|
| Control | Versión actual (A) que no recibe el cambio; sirve para saber qué habría pasado sin la intervención. | ¿Qué representa el punto de referencia?|
| Tratamiento | Versión nueva (B) cuyo efecto se quiere evaluar. | ¿Qué cambio recibe este grupo? | Entidad a la que se asigna la variante y que aporta una observación independiente. | ¿Sobre quién o qué se aplica la variante? |
| Variable respuesta | Resultado medido en cada unidad después de la asignación. | ¿Qué resultado se observa en cada unidad? | Indicador único, definido de antemano, que decide el éxito del experimento. | ¿Cuál será el criterio principal de éxito? |
| Guardrail | Métrica de protección que no debe empeorar más de un margen aceptable aunque la primaria mejore. | ¿Qué no debería deteriorarse? |
| MDE | Efecto mínimo detectable: la diferencia que se usa para dimensionar la muestra y calcular la potencia. Se fija antes de ver datos. | ¿Qué efecto se utiliza para planificar la potencia?|
| Potencia | Probabilidad (1−β) de rechazar H0 cuando el efecto verdadero es igual al MDE. | ¿Qué capacidad tiene el diseño para detectar el MDE? |
| Peeking | Mirar resultados parciales y detener, prolongar o decidir en función de ellos. | ¿Por qué no se debe decidir mirando resultados parciales?|

**Pausa de verificación:** explique a un compañero por qué el MDE no debe elegirse después de conocer la diferencia observada.



## Actividad 2 — Interpretación y representación metodológica
NexaCommerce, empresa ficticia, compara el checkout actual de cuatro pasos con uno simplificado de dos pasos. Cada usuario recibe una sola variante. Se mide compra e incidencia de pago dentro de siete días; incluso quienes no compran permanecen en el denominador.

### 2.1 Interpretación
Complete antes de ejecutar: Determinar si **el checkout simplificado de dos pasos (B) aumenta la proporción de usuarios que compran dentro de 7 días respecto del checkout actual de cuatro pasos (A) en más de 1,0 pp, sin que la incidencia de pago (guardrail) empeore en +0,5 pp o más**.

### 2.2 Representación del diseño
| Elemento | Definición en este experimento |
|---|---|
| Factor y niveles | Checkout: A actual; B simplificado |
| Unidad experimental | Usuario único con asignación persistente |
| Respuesta primaria | compra_7d: 1 si compró dentro de siete días; 0 si no |
| Métrica primaria | Usuarios que compraron / todos los usuarios asignados |
| Guardrail | incidencia_pago_7d: al menos una incidencia por usuario en siete días |
| Denominador del guardrail | Todos los usuarios asignados, compren o no |
| Margen de deterioro | B−A debe ser inferior a +0,5 pp en incidencias |
| Baseline histórico supuesto | Conversión 11,8 %; incidencias 1,8 % |
| MDE de conversión | +1,5 pp para planificar potencia de 80 % frente a cero |
| Umbral empresarial | Mejora de conversión superior a +1,0 pp |
| Horizonte | Reclutamiento planificado de 14 días y 7 días de maduración |
| Muestra | Se calcula antes de simular las respuestas |

### 2.3 Hipótesis y reglas previas
Conversión: H0: pB−pA ≤ 0; H1: pB−pA > 0. Alfa = 0,05 unilateral.

Relevancia empresarial: exigir límite inferior unilateral del 95 % mayor que 0,010. Esto evalúa un umbral distinto de cero y no se deduce solo de la estimación puntual.

Guardrail: H0: qB−qA ≥ 0,005; H1: qB−qA < 0,005. Exigir límite superior unilateral del 95 % menor que 0,005. No encontrar una diferencia significativa no demuestra que el guardrail esté protegido.

La recomendación combina ambos requisitos; se trata de una regla conjunta, no de escoger la prueba favorable. Los intervalos individuales no deben describirse como una región de confianza simultánea.

**Comprobación:** ¿por qué se comparan proporciones? ¿Cómo cambiarían las hipótesis si interesara cualquier diferencia, positiva o negativa?
**Respuesta:** Se comparan **proporciones** porque la variable respuesta es binaria (compró / no compró): el promedio de una variable 0/1 es una proporción, hay solo dos niveles del factor y cada usuario aporta una observación, por lo que corresponde una diferencia de dos proporciones (no ANOVA, que sirve para comparar medias de ≥3 grupos con respuesta continua). Si interesara **cualquier** diferencia, las hipótesis serían **H0: pB − pA = 0 vs H1: pB − pA ≠ 0**, es decir, un contraste **bilateral** con α/2 = 0,025 en cada cola (y el IC bilateral sería el intervalo coherente con la prueba).


## Actividad 3 — Preparación y planificación previa
### Paso 1. Preparar Google Colab
Ejecute las celdas en orden. La configuración y el cálculo de muestra aparecen antes de generar las respuestas. En Colab, la primera celda instala solo los paquetes que faltan. Si se solicita reiniciar, vuelva a ejecutar desde aquí.


In [ ]:
import importlib.util
import subprocess
import sys

paquetes = ["numpy", "pandas", "scipy", "statsmodels", "plotly"]
faltan = [p for p in paquetes if importlib.util.find_spec(p) is None]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
print("Dependencias disponibles.")


Dependencias disponibles.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

pd.set_option("display.max_columns", 20)
print("Entorno listo.")


Entorno listo.


### Paso 2. Fijar los parámetros antes de observar
Los baselines son supuestos históricos del caso ficticio, no estimaciones obtenidas de los grupos que analizaremos. El margen del guardrail se fija por criterio de negocio. Se calcula muestra para conversión y, por aproximación normal, para no inferioridad del guardrail suponiendo diferencia verdadera cero. Se toma el mayor tamaño. Las potencias se refieren a cada objetivo de diseño; no garantizan 80 % para la regla conjunta ni para superar el umbral empresarial.


In [ ]:
ALPHA = 0.05
POTENCIA = 0.80
BASELINE = 0.118
MDE = 0.015
UMBRAL_NEGOCIO = 0.010
BASE_GUARDRAIL = 0.018
MARGEN_GUARDRAIL = 0.005
SEMILLA = 20260921
DIAS_RECLUTAMIENTO = 14
VENTANA_DIAS = 7

assert 0 < BASELINE < BASELINE + MDE < 1
assert 0 < UMBRAL_NEGOCIO < MDE
analysis = NormalIndPower()
es = proportion_effectsize(BASELINE + MDE, BASELINE)
n_conversion = int(np.ceil(analysis.solve_power(
    effect_size=es, alpha=ALPHA, power=POTENCIA,
    ratio=1, alternative="larger")))
z_alpha = norm.ppf(1 - ALPHA)
z_power = norm.ppf(POTENCIA)
n_guardrail = int(np.ceil(
    2 * BASE_GUARDRAIL * (1 - BASE_GUARDRAIL)
    * (z_alpha + z_power)**2 / MARGEN_GUARDRAIL**2))
N_POR_GRUPO = max(n_conversion, n_guardrail)
print("Muestra por grupo para conversión:", n_conversion)
print("Muestra aproximada por grupo para guardrail:", n_guardrail)
print("Muestra planificada por grupo:", N_POR_GRUPO)
print("Analizar una sola vez al completar muestra y seguimiento.")


Muestra por grupo para conversión: 6027
Muestra aproximada por grupo para guardrail: 8743
Muestra planificada por grupo: 8743
Analizar una sola vez al completar muestra y seguimiento.


### Pausa de análisis 1
1. ¿Por qué el baseline debe establecerse antes de ver el resultado?
2. ¿Por qué la muestra final considera también el guardrail?
3. ¿La potencia calculada garantiza demostrar una mejora superior a 1,0 pp?

**RESPUESTA**

 1. **Baseline antes del resultado:** es un supuesto histórico de referencia que sirve para calcular el tamaño de muestra y la potencia. Si se tomara de los grupos analizados, el diseño dependería de los datos observados (circularidad, sesgo hacia el resultado deseado) y la muestra ya no serviría para planificar.
2. **Guardrail en la muestra:** porque el experimento debe poder responder *ambas* preguntas (mejora de conversión y no deterioro). Con incidencia base de 1,8 % y margen de solo 0,5 pp, el guardrail exige **8 743** usuarios por grupo frente a **6 027** de conversión; se toma el mayor (8 743, total 17 486) para que ninguna pregunta quede subdimensionada.
 3. **No.** La potencia de 80 % es la probabilidad de detectar un efecto verdadero de +1,5 pp frente a **cero**. Demostrar que el efecto supera **+1,0 pp** (límite inferior > 1,0 pp) es otra pregunta, con menos potencia: si el efecto verdadero fuera exactamente 1,5 pp, con n = 8 743 por grupo la probabilidad de que el límite inferior unilateral supere 1,0 pp sería de ~26 %. Tampoco garantiza 80 % para la regla conjunta.

### Paso 3. Generar y validar el conjunto académico
El simulador asigna aleatoriamente el mismo número de usuarios a A y B. Genera una fila por usuario y completa siete días de observación para todos. Supone conversiones de 11,8 % y 13,8 % e incidencias de 1,8 % en ambos grupos. Son parámetros del generador, no resultados que deban recuperarse exactamente. Las respuestas se simulan independientes para simplificar el ejercicio; una empresa real puede presentar dependencias y estacionalidad no representadas aquí.


In [ ]:
rng = np.random.default_rng(SEMILLA)
total = 2 * N_POR_GRUPO
grupos = rng.permutation(np.repeat(["A_Control", "B_Tratamiento"], N_POR_GRUPO))
prob_compra = np.where(grupos == "A_Control", 0.118, 0.138)
df = pd.DataFrame({
    "usuario_id": np.arange(1, total + 1),
    "grupo": grupos,
    "dia_asignacion": rng.integers(1, DIAS_RECLUTAMIENTO + 1, total),
    "dias_observados": np.full(total, VENTANA_DIAS),
    "compra_7d": rng.binomial(1, prob_compra),
    "incidencia_pago_7d": rng.binomial(1, BASE_GUARDRAIL, total)
})
print("DATOS SIMULADOS — NexaCommerce")
print("Dimensión:", df.shape)
display(df.head())


DATOS SIMULADOS — NexaCommerce
Dimensión: (17486, 6)


,usuario_id,grupo,dia_asignacion,dias_observados,compra_7d,incidencia_pago_7d
0,1,A_Control,8,7,0,0
1,2,B_Tratamiento,5,7,0,0
2,3,A_Control,12,7,0,0
3,4,A_Control,9,7,0,0
4,5,B_Tratamiento,12,7,0,0


In [ ]:
requeridas = {"usuario_id", "grupo", "dias_observados",
              "compra_7d", "incidencia_pago_7d"}
assert requeridas.issubset(df.columns), "Faltan columnas"
assert not df[list(requeridas)].isna().any().any(), "Hay datos ausentes"
assert df["usuario_id"].is_unique, "Usuarios repetidos o contaminados"
assert set(df["grupo"]) == {"A_Control", "B_Tratamiento"}
assert (df["dias_observados"] == VENTANA_DIAS).all()
for col in ["compra_7d", "incidencia_pago_7d"]:
    assert df[col].isin([0, 1]).all(), f"{col} no es binaria"
ab = df.copy()
tam_grupos = ab.groupby("grupo").size()
assert (tam_grupos == N_POR_GRUPO).all()
display(tam_grupos.to_frame("usuarios"))
print("Calidad estructural y seguimiento completos.")


,usuarios
grupo,
A_Control,8743
B_Tratamiento,8743


Calidad estructural y seguimiento completos.


### Pausa de análisis 2
1. Registre el tamaño de cada grupo. ¿Cumple el plan?
2. ¿Qué ocurriría si un usuario apareciera en ambas variantes?
3. ¿Por qué no debe excluirse a quienes no compraron?
4. ¿Por qué es necesario completar los siete días antes de analizar?

**RESPUESTA**

1. Grupo A_Control: **8 743** usuarios; grupo B_Tratamiento: **8 743** usuarios (total 17 486). **Sí cumple** el plan (N_POR_GRUPO = 8 743, grupos balanceados).
2. Si un usuario apareciera en ambas variantes habría **contaminación**: recibiría A y B, ya no sería una unidad con asignación persistente, su respuesta no se podría atribuir a una sola variante, se violaría la independencia entre grupos y el efecto se diluiría o sesgaría. La validación de `usuario_id` único lo detectaría y habría que corregir el diseño o excluir/reasignar según el plan, no "arreglarlo" viendo resultados.
3. Porque la métrica es la proporción sobre **todos los usuarios asignados**. Quitar a quienes no compraron dejaría solo compradores (conversión artificial de 100 %), eliminaría justo lo que se quiere medir y generaría sesgo de selección posterior a la asignación.
4. Para que **todos** los usuarios tengan la misma ventana de oportunidad de comprar. Quienes entraron el día 14 tendrían menos días que los del día 1, subestimando la conversión (y de forma potencialmente distinta entre grupos). Completar la ventana también evita el *peeking*.

## Actividad 4 — MDE, potencia y sensibilidad de muestra
### Paso 4. Recuperar el diseño fijado
Revise los parámetros sin recalibrarlos con el resultado observado. El MDE sirve para dimensionar la muestra; el umbral empresarial se utiliza para valorar la mejora. En este caso, las dos cantidades difieren deliberadamente.


In [ ]:
display(pd.DataFrame({
    "parametro": ["Baseline", "MDE", "Umbral empresarial",
                  "Margen guardrail", "Alfa", "Potencia"],
    "valor": [BASELINE, MDE, UMBRAL_NEGOCIO,
              MARGEN_GUARDRAIL, ALPHA, POTENCIA]
}))
print("Muestra final por grupo:", N_POR_GRUPO)


,parametro,valor
0,Baseline,0.118
1,MDE,0.015
2,Umbral empresarial,0.010
3,Margen guardrail,0.005
4,Alfa,0.050
5,Potencia,0.800


Muestra final por grupo: 8743


### Paso 5. Explorar la sensibilidad al MDE
Compare MDE de 0,5, 1,5 y 2,5 pp. Esta exploración es didáctica y no cambia el plan ni la decisión del caso ya fijado.
**Predicción:** ¿qué efecto necesitará más muestra y por qué?


In [ ]:
filas = []
for mde in [0.005, 0.015, 0.025]:
    es = proportion_effectsize(BASELINE + mde, BASELINE)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA,
                             power=POTENCIA, alternative="larger")
    filas.append({"MDE_pp": mde * 100,
                  "n_conversion": int(np.ceil(n)),
                  "n_con_guardrail": max(int(np.ceil(n)), n_guardrail)})
sensibilidad_n = pd.DataFrame(filas)
display(sensibilidad_n)
fig = px.line(sensibilidad_n, x="MDE_pp",
              y=["n_conversion", "n_con_guardrail"], markers=True,
              title="Muestra por grupo según el MDE",
              labels={"value": "Usuarios por grupo", "MDE_pp": "MDE (pp)"})
fig.show()


,MDE_pp,n_conversion,n_con_guardrail
0,0.5,52413,52413
1,1.5,6027,8743
2,2.5,2241,8743


### Pausa de análisis 3
1. Explique por qué reducir el MDE aumenta la muestra para conversión.
2. ¿Por qué el guardrail puede determinar la muestra final?
3. ¿Sería válido aumentar el MDE después de ver los resultados para declarar éxito?

**RESPUESTA**

**Predicción (Paso 5):** el MDE de **0,5 pp** necesita más muestra, porque es el efecto más pequeño y más difícil de separar del ruido. Resultado de la celda: 0,5 pp → 52 413 por grupo; 1,5 pp → 6 027 (8 743 con guardrail); 2,5 pp → 2 241 (8 743 con guardrail).

1. La muestra crece aproximadamente con 1/(efecto)²: un efecto menor produce una diferencia más pequeña frente a un error estándar que solo baja con √n, así que hace falta muchísima más muestra (triplicar el MDE reduce n unas 9 veces; pasar de 1,5 a 0,5 pp lo multiplica ≈ 8,7×).
2. Porque el guardrail se mide con una incidencia baja (1,8 %) y un margen estrecho (0,5 pp), lo que exige n = 8 743. Cuando el MDE de conversión es grande (1,5 o 2,5 pp), el guardrail domina (n_con_guardrail = 8 743 en ambos casos).
3. **No.** Aumentar el MDE después de ver que la diferencia observada fue grande sería ajustar el diseño al resultado (HARKing / *post hoc*): el MDE debe ser previo y no cambia la potencia realmente planificada ni la regla de decisión.

## Actividad 5 — Análisis estadístico de la comparación A/B

**Propósito:** calcular el efecto observado y evaluar la hipótesis mediante una prueba para dos proporciones.

### Paso 6. Obtener tasas y tamaño del efecto observado

Antes de ejecutar, recuerde:

- Diferencia absoluta: $p_B-p_A$.
- Lift relativo: $(p_B-p_A)/p_A$.
- La diferencia se expresa en puntos porcentuales; el lift expresa un cambio relativo respecto de A.


In [ ]:
tabla = ab.groupby("grupo")["compra_7d"].agg(["sum", "count", "mean"])
tabla.columns = ["compras", "n", "tasa_conversion"]
x_A, n_A = int(tabla.loc["A_Control", "compras"]), int(tabla.loc["A_Control", "n"])
x_B, n_B = int(tabla.loc["B_Tratamiento", "compras"]), int(tabla.loc["B_Tratamiento", "n"])
p_A, p_B = x_A / n_A, x_B / n_B
efecto = p_B - p_A
lift_rel = efecto / p_A if p_A > 0 else np.nan
display(tabla)
print(f"Diferencia B−A: {efecto*100:.3f} pp")
print(f"Lift relativo: {lift_rel:.2%}")


,compras,n,tasa_conversion
grupo,,,
A_Control,996,8743,0.11392
B_Tratamiento,1236,8743,0.14137


Diferencia B−A: 2.745 pp
Lift relativo: 24.10%


### Pausa de análisis 4

Complete sin utilizar todavía el p-valor:

La tasa de B es **14,14 %** (1 236/8 743) y la tasa de A es **11,39 %** (996/8 743). La diferencia observada es **2,745** puntos porcentuales, equivalente a un lift relativo de **24,10 %**. Estos valores describen la muestra, pero todavía no permiten afirmar que **la mejora sea real en la población y no se deba solo a la variabilidad del muestreo, ni que supere el umbral empresarial de 1,0 pp con evidencia suficiente** (para eso se necesita el contraste y los intervalos).


### Paso 7. Contraste e intervalos coherentes
Usamos la aproximación normal de Wald con error estándar no combinado. Se comprueba que haya al menos diez compras y diez no compras en cada grupo; esta regla práctica no sustituye la revisión del diseño.

Para la prueba unilateral frente a cero, p < 0,05 equivale a límite inferior unilateral del 95 % > 0 con el mismo método. El IC bilateral del 95 % se presenta aparte. La función también permite contrastar un umbral distinto de cero. El p-valor no es la probabilidad de que H0 sea verdadera.


In [ ]:
def diferencia_wald(x_b, n_b, x_a, n_a, umbral=0.0):
    if min(x_b, n_b-x_b, x_a, n_a-x_a) < 10:
        raise ValueError("Frecuencias pequeñas: revisar método antes de continuar")
    pb, pa = x_b/n_b, x_a/n_a
    d = pb-pa
    se = np.sqrt(pb*(1-pb)/n_b + pa*(1-pa)/n_a)
    z = (d-umbral)/se
    return {"d": d, "se": se, "z": z,
            "p_mayor": norm.sf(z), "p_menor": norm.cdf(z),
            "li_uni95": d-norm.ppf(1-ALPHA)*se,
            "ls_uni95": d+norm.ppf(1-ALPHA)*se,
            "ic95_low": d-norm.ppf(1-ALPHA/2)*se,
            "ic95_high": d+norm.ppf(1-ALPHA/2)*se}

conv = diferencia_wald(x_B, n_B, x_A, n_A)
negocio = diferencia_wald(x_B, n_B, x_A, n_A, UMBRAL_NEGOCIO)
print(f"z frente a cero: {conv['z']:.4f}")
print(f"p unilateral frente a cero: {conv['p_mayor']:.6g}")
print(f"Límite inferior unilateral 95%: {conv['li_uni95']*100:.3f} pp")
print(f"IC bilateral 95%: [{conv['ic95_low']*100:.3f}, {conv['ic95_high']*100:.3f}] pp")
print(f"p unilateral frente a +1,0 pp: {negocio['p_mayor']:.6g}")


z frente a cero: 5.4436
p unilateral frente a cero: 2.61091e-08
Límite inferior unilateral 95%: 1.916 pp
IC bilateral 95%: [1.757, 3.733] pp
p unilateral frente a +1,0 pp: 0.000269553


In [ ]:
grafico = tabla.reset_index()
grafico["conversion_pct"] = grafico["tasa_conversion"] * 100
fig = px.bar(grafico, x="grupo", y="conversion_pct", text="conversion_pct",
             title="Conversión observada — datos simulados",
             labels={"conversion_pct": "Usuarios que compraron (%)", "grupo": "Grupo"})
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_yaxes(rangemode="tozero")
fig.show()


### Análisis y conclusión estadística
1. Registre alfa, z y p unilateral frente a cero.
2. ¿Se rechaza H0? Explique en el contexto de conversión del caso simulado.
3. ¿El límite inferior unilateral supera cero? ¿Y el umbral de 1,0 pp?
4. Explique por qué el IC bilateral del 95 % y el límite unilateral del 95 % tienen extremos distintos.
5. ¿Superar 1,0 pp con la estimación puntual demuestra superar ese umbral con evidencia suficiente?

 **RESPUESTA**

 1. **α = 0,05** (unilateral); **z = 5,4436** (error estándar no combinado = 0,504 pp); **p unilateral = 2,6 × 10⁻⁸** (≈ 0,00000003).
2. **Sí, se rechaza H0** (p < 0,05): en este caso simulado hay evidencia estadística de que la conversión de B es mayor que la de A (pB − pA > 0). No dice qué tan grande es el efecto ni que sea relevante para el negocio.
 3. **Límite inferior unilateral 95 % = 1,916 pp**: supera **cero** y también el umbral de **+1,0 pp** (p unilateral frente a +1,0 pp = 0,00027).
 4. Porque usan cuantiles distintos con el mismo error estándar: el IC bilateral deja 2,5 % en cada cola (z = 1,96) y da [1,757 ; 3,733] pp; el límite unilateral deja 5 % en una sola cola (z = 1,645), por lo que su extremo inferior (1,916 pp) es más alto y el superior queda abierto. Responden preguntas distintas: descripción de la incertidumbre en ambos lados vs. cota mínima del efecto.
 5. **No.** La estimación puntual (2,745 pp) es solo el centro de la incertidumbre. Para afirmar que se supera el umbral hay que mostrar que el **límite inferior** lo excede (1,916 > 1,0) o que la prueba frente al umbral rechaza (p = 0,00027). Aquí sí ocurre, pero no por la estimación puntual sola.

## Actividad 6 — Del resultado estadístico a la decisión
### Paso 8. Evaluar el guardrail con incertidumbre
El guardrail es la proporción de usuarios con alguna incidencia de pago en siete días. El margen de deterioro admisible es +0,5 pp. La no inferioridad exige que el límite superior unilateral del 95 % esté por debajo de ese margen. No basta obtener p > 0,05 en una prueba de igualdad.


In [ ]:
guardrail = ab.groupby("grupo")["incidencia_pago_7d"].agg(["sum", "count", "mean"])
guardrail.columns = ["incidencias", "n", "tasa_incidencia"]
g_A = int(guardrail.loc["A_Control", "incidencias"])
g_B = int(guardrail.loc["B_Tratamiento", "incidencias"])
guard = diferencia_wald(g_B, n_B, g_A, n_A, MARGEN_GUARDRAIL)
display(guardrail)
print(f"Cambio del guardrail B−A: {guard['d']*100:.3f} pp")
print(f"Límite superior unilateral 95%: {guard['ls_uni95']*100:.3f} pp")
print(f"Margen permitido: {MARGEN_GUARDRAIL*100:.1f} pp")
print(f"p de no inferioridad: {guard['p_menor']:.6g}")


,incidencias,n,tasa_incidencia
grupo,,,
A_Control,139,8743,0.015898
B_Tratamiento,176,8743,0.020130


Cambio del guardrail B−A: 0.423 pp
Límite superior unilateral 95%: 0.754 pp
Margen permitido: 0.5 pp
p de no inferioridad: 0.351286


### Paso 9. Integrar la regla de decisión
| Criterio | Evidencia requerida |
|---|---|
| Diseño y calidad | Una fila por usuario, asignación persistente, muestra y seguimiento completos |
| Mejora frente a cero | p unilateral < 0,05 |
| Relevancia empresarial | Límite inferior unilateral del 95 % > +1,0 pp |
| Guardrail | Límite superior unilateral del 95 % < +0,5 pp |

La regla se fijó antes de simular. El MDE dimensiona la muestra y no reemplaza al umbral empresarial. Si no se satisface un requisito, distinguir evidencia insuficiente de evidencia de perjuicio. Cualquier experimento posterior necesita un nuevo plan; no seguir agregando datos hasta lograr significancia.


In [ ]:
cumple_estadistica = bool(conv["p_mayor"] < ALPHA)
cumple_negocio = bool(conv["li_uni95"] > UMBRAL_NEGOCIO)
cumple_guardrail = bool(guard["ls_uni95"] < MARGEN_GUARDRAIL)
cumple_diseno = bool((tam_grupos >= N_POR_GRUPO).all())
criterios = pd.Series({"Diseño validado": cumple_diseno,
    "Mejora frente a cero": cumple_estadistica,
    "Mejora superior al umbral": cumple_negocio,
    "Guardrail protegido": cumple_guardrail})
display(criterios.to_frame("cumple"))
if criterios.all():
    decision = "Recomendar B en el caso simulado, con seguimiento posterior."
else:
    decision = "No recomendar el despliegue todavía: revisar los criterios no cumplidos."
print(decision)
print("Conclusión académica con datos simulados; no valida un despliegue real.")


,cumple
Diseño validado,True
Mejora frente a cero,True
Mejora superior al umbral,True
Guardrail protegido,False


No recomendar el despliegue todavía: revisar los criterios no cumplidos.
Conclusión académica con datos simulados; no valida un despliegue real.


### Pausa de decisión
| Componente | Resultado observado | Interpretación |
|---|---|---|
| p unilateral frente a cero | 2,6 × 10⁻⁸ (z = 5,44) | Se rechaza H0: hay evidencia fuerte de que B convierte más que A en este caso simulado. |
| Diferencia B−A en pp y lift relativo | +2,745 pp (14,14 % vs 11,39 %); lift 24,10 % | Efecto observado mayor que el MDE (1,5 pp) y que el umbral (1,0 pp); es un valor de la muestra, no una garantía poblacional. |
| Límite inferior y umbral empresarial | LI unilateral 95 % = 1,916 pp > 1,0 pp (p frente a 1,0 pp = 0,00027) | Hay evidencia suficiente de mejora superior al umbral empresarial. IC bilateral 95 %: [1,757 ; 3,733] pp. |
| Límite superior del guardrail y margen | B−A = +0,423 pp (2,01 % vs 1,59 %); LS unilateral 95 % = 0,754 pp > 0,5 pp (p no inferioridad = 0,351) | **No** se demuestra que el guardrail esté protegido: el deterioro plausible puede llegar a 0,75 pp, más que el margen. Es evidencia insuficiente, no prueba de perjuicio. |
| Calidad y límites del caso simulado | Grupos de 8 743 c/u, sin ausentes, usuarios únicos, 7 días completos | Diseño válido y seguimiento completo, pero datos sintéticos, respuestas independientes, sin estacionalidad; la conclusión es académica y no valida un despliegue real. |

**Decisión en cuatro líneas:** **no recomendar todavía**, porque el guardrail no queda protegido (ver respuesta). Incluya al menos dos cifras. Si falta evidencia, proponga un nuevo diseño válido; no continúe esta muestra buscando significancia.

 **RESPUESTA**

 **No recomendar el despliegue de B todavía**, porque aunque la conversión mejora con evidencia sólida (+2,745 pp, LI unilateral = 1,916 pp > 1,0 pp; p = 2,6 × 10⁻⁸), el guardrail no queda protegido (LS unilateral = 0,754 pp > 0,5 pp).
 La regla conjunta exige los cuatro criterios y el resultado es **[Diseño ✔, Mejora vs cero ✔, Umbral empresarial ✔, Guardrail ✘]**.
Esto es **evidencia insuficiente** sobre incidencias (B−A = +0,423 pp, p de no inferioridad = 0,351), no evidencia de perjuicio superior al margen.
 **Propuesta:** un nuevo experimento con plan previo que reduzca la incertidumbre del guardrail (p. ej., ≈ 12 110 usuarios por grupo para 90 % de potencia de no inferioridad con la misma base y margen, o revisar formalmente el margen con negocio *antes* de recoger datos) y monitoreo de incidencias; no se debe seguir agregando datos a esta muestra ni cambiar la semilla.

 *Nota de rigor:* el IC bilateral 95 % de la diferencia de incidencias es ≈ [0,03 ; 0,82] pp, así que ronda el cero por abajo; eso no era una prueba prerregistrada y en el simulador ambos grupos tienen la misma incidencia verdadera (1,8 %), por lo que la diferencia observada es ruido de muestreo. Aun así, con los datos observados la regla previa impide afirmar que el guardrail está protegido.

## Actividad 7 — Retos de aplicación y retroalimentación
Se resuelven durante la clase y no generan una entrega adicional.

### Reto 1 — Estimación y umbral
Un caso presenta p = 0,02 frente a cero, mejora estimada de 0,4 pp y umbral empresarial de 3 pp. ¿Qué demuestra la prueba y qué no demuestra? ¿Qué información aporta el intervalo?
**Respuesta:**

 La prueba (p = 0,02 frente a cero) demuestra que hay evidencia de que el efecto es **mayor que cero** (sería poco probable ver esa diferencia si el efecto verdadero fuera ≤ 0); **no** demuestra que sea grande ni relevante: la mejora estimada (0,4 pp) está muy por debajo del umbral empresarial de 3 pp, así que hay significancia estadística sin relevancia práctica. El p-valor tampoco es la probabilidad de que H0 sea cierta. El intervalo aporta el rango plausible del efecto: con estimación 0,4 pp su límite inferior estará por encima de 0 pero lejos de 3 pp, y por tanto no hay evidencia de superar el umbral; permite juzgar magnitud e incertidumbre, no solo "significativo / no significativo".

### Reto 2 — Cambio de pregunta
Si interesa cualquier diferencia entre A y B, formule H0 y H1 e indique el tipo de contraste. ¿Por qué debe elegirse antes de observar los resultados?
**Respuesta:**

**H0: pB − pA = 0** vs **H1: pB − pA ≠ 0**, con un contraste **bilateral** (α/2 = 0,025 por cola). Debe elegirse antes de observar porque elegir la dirección o el tipo de prueba después de ver el signo de la diferencia duplica la probabilidad real de falso positivo (p-hacking): se escogería la cola que favorece el resultado.

### Reto 3 — Guardrail no concluyente
El margen es +0,5 pp. El aumento estimado de incidencias es +0,2 pp, pero el límite superior unilateral es +0,7 pp. ¿Se ha demostrado cumplir el guardrail? ¿Es correcto afirmar que se ha demostrado perjuicio?
**Respuesta:**

 **No**, no se ha demostrado el cumplimiento del guardrail: se exige LS unilateral < +0,5 pp y aquí es +0,7 pp; la estimación puntual (+0,2 pp) no basta. Tampoco es correcto afirmar que se demostró perjuicio: para eso el límite **inferior** debería superar el margen (o al menos 0 con relevancia). El resultado es **no concluyente / evidencia insuficiente**; no demuestra igualdad ni daño. Se necesitaría más precisión con un nuevo diseño planificado.

### Reto 4 — Reproducibilidad y sesgo
Un equipo cambia la semilla hasta conseguir una decisión favorable. Explique por qué eso invalida el ejercicio. Proponga cómo documentar una exploración de escenarios sin sustituir el caso base.
**Respuesta:**

 Cambiar la semilla hasta obtener una decisión favorable es seleccionar resultados (equivale a repetir el experimento y quedarse con el que conviene): aumenta la tasa de falsos positivos, invalida el p-valor y viola el prerregistro y la trazabilidad. **Cómo documentar escenarios sin reemplazar el caso base:** conservar la semilla 20260921 como caso base y su decisión; declarar *antes* una lista fija de semillas o escenarios (p. ej., 100 semillas o distintos supuestos de conversión/incidencia), ejecutar todas, registrar semilla, parámetros y versión del código, y reportar la distribución completa (proporción de veces que se cumple cada criterio, rangos de p y de límites) en una sección exploratoria claramente separada.


# VIII. Secuencia de trabajo para la sesión
| Momento | Minutos | Desarrollo |
|---|---:|---|
| Apertura | 10 | Pregunta orientadora y caso |
| Actividad 1 | 15 | Activación conceptual |
| Actividad 2 | 20 | Representación e hipótesis |
| Actividad 3 | 30 | Planificación, simulación y calidad |
| Receso | 15 | Pausa |
| Actividad 4 | 25 | Sensibilidad de muestra |
| Actividad 5 | 30 | Tasas, contraste e intervalos |
| Actividad 6 | 20 | Guardrail y decisión |
| Actividad 7 | 15 | Retos y ticket de salida |

Total: 180 minutos, incluido el receso.

## Ticket de salida
«Un experimento A/B no termina en el p-valor porque…».

 **RESPUESTA**

 «Un experimento A/B no termina en el p-valor porque el p-valor solo indica si el resultado es compatible con H0; no informa el **tamaño** del efecto, su **incertidumbre** (intervalos), su **relevancia práctica** frente al umbral empresarial, ni si los **guardrails** y la calidad del diseño (aleatorización, muestra completa, sin peeking) sostienen la decisión.» En este caso: p = 2,6 × 10⁻⁸ y LI = 1,92 pp > 1,0 pp, pero el guardrail (LS = 0,75 pp > 0,5 pp) impide recomendar B todavía.

# IX. Lista de verificación de aprendizaje
- [x] Identifico factor, niveles y unidad experimental.
- [x] Defino respuesta, denominador y ventana de observación.
- [x] Formulo hipótesis antes de observar resultados.
- [x] Distingo baseline, MDE, umbral empresarial, alfa y potencia.
- [x] Calculo muestra antes de generar los datos del caso.
- [x] Interpreto pp, lift, p-valor e intervalos sin confundirlos.
- [x] Evalúo el guardrail con margen e incertidumbre.
- [x] Distingo evidencia insuficiente de evidencia de perjuicio.
- [x] Reconozco peeking, contaminación y selección de resultados.
- [x] Declaro el carácter simulado y los límites del ejercicio.


# X. Referencias
- Kohavi, R., Tang, D., y Xu, Y. (2020). Trustworthy Online Controlled Experiments. Cambridge University Press.
- Bruce, P., Bruce, A., y Gedeck, P. (2020). Practical Statistics for Data Scientists, segunda edición. O’Reilly.
- statsmodels. Documentación de NormalIndPower y proportion_effectsize: https://www.statsmodels.org/stable/stats.html
- SciPy. Distribución normal, funciones cdf, sf y ppf: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html

Datos: simulación académica NexaCommerce incluida en el notebook, semilla 20260921. Los datos no proceden de las referencias bibliográficas.
